# 01 - EDA BraTS 2020 NIfTI

Exploration des modalités MRI, des dimensions et des masques.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = REPO_ROOT / "data/raw/brats2020-training-data"
FIGURE_ROOT = REPO_ROOT / "results/figures/eda"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
files = sorted(list(DATA_ROOT.rglob("*.nii.gz")) + list(DATA_ROOT.rglob("*.nii")))
print(f"Dataset: {DATA_ROOT}")
print(f"Fichiers NIfTI: {len(files)}")

Dataset: c:\Users\Dell\Desktop\brain-mri-fem-deep-learning\data\raw\brats2020-training-data
Fichiers NIfTI: 0


In [1]:
def modality(path: Path) -> str:
    name = path.name.lower().replace(".nii.gz", "").replace(".nii", "")
    return next((suffix[1:] for suffix in ("_flair", "_t1ce", "_t1", "_t2", "_seg") if name.endswith(suffix)), "unknown")

def patient_id(path: Path) -> str:
    name = path.name.lower().replace(".nii.gz", "").replace(".nii", "")
    suffix = f"_{modality(path)}"
    return name[:-len(suffix)] if name.endswith(suffix) else name

if not files:
    print("Aucun fichier trouvé. Téléchargez BraTS puis relancez les cellules suivantes.")
else:
    table = pd.DataFrame({"path": files})
    table["patient"] = table["path"].map(patient_id)
    table["modality"] = table["path"].map(modality)
    display(table.groupby("modality").size().rename("count").to_frame())
    display(table.head())

NameError: name 'files' is not defined

In [ ]:
if files:
    rows = []
    for path in files:
        image = nib.load(str(path))
        rows.append({"patient": patient_id(path), "modality": modality(path), "shape": tuple(image.shape), "spacing": tuple(image.header.get_zooms()[:3]), "orientation": "".join(nib.aff2axcodes(image.affine))})
    metadata = pd.DataFrame(rows)
    display(metadata.groupby("modality")[['shape', 'spacing', 'orientation']].first())
    metadata.to_csv(FIGURE_ROOT / "volume_metadata.csv", index=False)
    sample = table[table["modality"] == "flair"].iloc[0]["path"]
    volume = np.asarray(nib.load(str(sample)).get_fdata(), dtype=np.float32)
    index = volume.shape[2] // 2
    plt.figure(figsize=(5, 5))
    plt.imshow(volume[:, :, index].T, cmap="gray", origin="lower")
    plt.title(f"FLAIR - coupe {index}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(FIGURE_ROOT / "flair_middle_slice.png", dpi=150)
    plt.show()